# IFEST 2026 DAC — V2 Global + Local MIL NLI

**Task:** headline–article factual consistency (`1 = sesuai`, `0 = tidak sesuai`), evaluated with **Macro F1**.

This notebook is the targeted V2 iteration after the V1.1.1 public score of **0.93367**. It intentionally preserves the components that already worked and changes only the residual bottlenecks diagnosed from V1 OOF errors.

V2 design:

1. **Same mDeBERTa-v3 multilingual NLI backbone** that was already proven in V1.
2. **Connected-component Stratified Group CV** remains the strict semantic validation scheme.
3. **Global + Local Multi-Instance NLI:** one compact global evidence view plus three independent local evidence spans per row.
4. **Partial NLI supervision:** positives maximize entailment; natural negatives maximize neutral-or-contradiction without forcing every negative into contradiction.
5. **Hardest real same-content counterfactual ranking** instead of arbitrary opposite-title pairing.
6. **Small, conservative counterfactual curriculum** for relation-sensitive mistakes (polarity / number / entity-like replacements), explicitly supervised as contradiction and capped so synthetic data cannot dominate.
7. **Fold-specific probability calibration** before test-fold averaging.
8. **Compact relational meta-calibrator** using full NLI state, local/global disagreement, and factual consistency measured on the model-selected support span.
9. **Safer graph layer:** exact-pair decisions, unique-canonical-positive content mismatch only, malformed/ambiguous-content abstention, and confidence-gated same-title mismatch.
10. **Full diagnostics and artifacts:** strict OOF, per-class F1, slice metrics, graph-like LOO diagnostic, training history, test scores, and final submission.

Default budget is still **3 folds × 2 epochs on T4×2**. Do not change to 5 folds for the first V2 run.

In [1]:
# Environment bootstrap. Kaggle normally already contains these packages.
import os, sys, subprocess, importlib.util

required = ["transformers", "sentencepiece", "sklearn", "pandas", "torch"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "transformers>=4.45,<6", "sentencepiece", "accelerate", "scikit-learn", "pandas"
    ])
    print("Installed missing packages:", missing)
else:
    print("Environment ready. Missing packages:", missing)

Environment ready. Missing packages: []


In [2]:
import os, re, gc, json, math, random, hashlib, unicodedata, warnings, time, difflib
from pathlib import Path
from collections import Counter, defaultdict
from contextlib import nullcontext

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedGroupKFold, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import transformers

warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")
pd.set_option("display.max_colwidth", 180)

# ---------------- V2 CONFIG ----------------
SEED = 2026
MODEL_NAME = "MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7"
LOCAL_MODEL_PATH = None  # optionally point this to a Kaggle Input containing the HF checkpoint

# Validation/training budget stays identical in spirit to the proven V1 run.
N_FOLDS = 3
EPOCHS = 2

# Global + local evidence design.
GLOBAL_MAX_LENGTH = 384
LOCAL_MAX_LENGTH = 224
GLOBAL_WINDOW_WORDS = 90
GLOBAL_WINDOW_STRIDE = 45
GLOBAL_TOP_WINDOWS = 3
LOCAL_WINDOW_WORDS = 150
LOCAL_WINDOW_STRIDE = 75
N_LOCAL_VIEWS = 3           # lead + two strongest non-identical local spans
MIL_POOL_ALPHA = 8.0
GLOBAL_LOSS_WEIGHT = 0.35
LOCAL_LOSS_WEIGHT = 0.65

# T4x2-oriented throughput. Global and local passes are intentionally separated to limit activation memory.
BATCH_SIZE = 6              # global batch; local forward sees <= 18 main local sequences
EVAL_BATCH_SIZE = 12
GRAD_ACCUM = 5              # effective primary batch ~= 30
NUM_WORKERS = 2
USE_AMP = True
USE_GRAD_CHECKPOINTING = False

LR = 1.5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08
NEG_CLASS_WEIGHT = 3.0      # preserve the V1 choice; do not jump to the raw ~9x imbalance ratio
MAX_GRAD_NORM = 1.0

# Real hardest counterfactual ranking.
PAIRWISE_LAMBDA = 0.25
PAIRWISE_MARGIN = 0.50
MAX_PAIRWISE_PER_BATCH = 3

# Conservative synthetic counterfactual curriculum. It is deliberately small.
USE_SYNTHETIC_COUNTERFACTUALS = True
SYNTHETIC_RATE = 0.15
SYNTHETIC_LAMBDA = 0.10
MAX_SYNTHETIC_PER_BATCH = 2

# Compact factual feature configuration.
HIGH_IDF_QUANTILE = 0.80

# Graph V2: exact -> unique canonical positive content -> gated same-title mismatch -> semantic/meta.
UNIQUE_CONTENT_RULE_MIN_PRECISION = 0.995
TITLE_GATE_MIN_PRECISION = 0.90
MIN_GRAPH_CONTENT_WORDS = 20

SAVE_FOLD_MODELS = False
OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "/mnt/data/ifest_v2_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
GPU_COUNT = torch.cuda.device_count()
AMP_ENABLED = bool(USE_AMP and DEVICE.type == "cuda")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print(f"PyTorch={torch.__version__} | Transformers={transformers.__version__}")
print(f"Device={DEVICE} | GPUs={GPU_COUNT} | AMP={AMP_ENABLED}")
if GPU_COUNT:
    for i in range(GPU_COUNT):
        props = torch.cuda.get_device_properties(i)
        print(i, torch.cuda.get_device_name(i), f"VRAM={props.total_memory/1024**3:.1f} GB")
if DEVICE.type != "cuda":
    print("WARNING: V2 is designed for Kaggle T4x2; CPU execution is not intended for full training.")

PyTorch=2.10.0+cu128 | Transformers=5.0.0
Device=cuda | GPUs=2 | AMP=True
0 Tesla T4 VRAM=14.6 GB
1 Tesla T4 VRAM=14.6 GB


In [3]:
def discover_data_files():
    roots = []
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
    if Path("/mnt/data/ifest_dac").exists():
        roots.append(Path("/mnt/data/ifest_dac"))

    train_candidates, test_candidates, sample_candidates = [], [], []
    for root in roots:
        train_candidates.extend(root.rglob("train.csv"))
        test_candidates.extend(root.rglob("test.csv"))
        sample_candidates.extend(root.rglob("sample_submission.csv"))

    def valid(path, required_cols):
        try:
            return set(required_cols).issubset(pd.read_csv(path, nrows=2).columns)
        except Exception:
            return False

    trains = [p for p in train_candidates if valid(p, ["id", "title", "content", "label"])]
    tests = [p for p in test_candidates if valid(p, ["id", "title", "content"])]
    samples = [p for p in sample_candidates if valid(p, ["id", "label"])]
    if not trains or not tests:
        raise FileNotFoundError("Could not locate IFEST train.csv/test.csv. Add the competition data as a Kaggle Input.")

    train_path = trains[0]
    same_test = [p for p in tests if p.parent == train_path.parent]
    test_path = same_test[0] if same_test else tests[0]
    same_sample = [p for p in samples if p.parent == train_path.parent]
    sample_path = same_sample[0] if same_sample else (samples[0] if samples else None)
    return train_path, test_path, sample_path

train_path, test_path, sample_path = discover_data_files()
train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
if sample_path is not None:
    sample = pd.read_csv(sample_path)
else:
    sample = pd.DataFrame({"id": test["id"], "label": 1})

assert list(train.columns)[:3] == ["id", "title", "content"]
assert {0, 1} == set(map(int, train.label.unique()))
assert train.id.is_unique and test.id.is_unique
assert len(sample) == len(test)

print("train:", train.shape, train_path)
print("test :", test.shape, test_path)
print("labels:")
print(train.label.value_counts().sort_index())
print("minority ratio:", round(float((train.label == 0).mean()), 4))

train: (14397, 4) /kaggle/input/competitions/penyisihan-ifest-2026-dac/train.csv
test : (3603, 3) /kaggle/input/competitions/penyisihan-ifest-2026-dac/test.csv
labels:
label
0     1437
1    12960
Name: count, dtype: int64
minority ratio: 0.0998


## Conservative normalization + global/local evidence views

V1 degradation concentrated in long articles and high-retrieval-gain rows. V2 therefore does **not** concatenate all evidence into one sequence and hope the most important span survives truncation.

Each row gets:

- a **global compact view** that keeps the article lead and puts the strongest retrieved windows early,
- three **local views** scored independently by the NLI model: article lead + two strongest relevant spans.

Numbers, negation, names, and punctuation-bearing numeric expressions are preserved in model text.

In [4]:
TOKEN_RE = re.compile(r"[\w]+", flags=re.UNICODE)
NUM_RAW_RE = re.compile(r"\b\d+(?:[.,]\d+)*\b")
SOURCE_SUFFIX_RE = re.compile(r"\s+kumparan\s+com\s*$", re.I)
NEGATIONS = {
    "tidak", "tak", "bukan", "belum", "tanpa", "jangan", "gagal", "mustahil",
    "dilarang", "menolak", "negatif", "tiada", "kurang"
}
POLARITY_WORDS = NEGATIONS | {
    "sudah", "positif", "naik", "turun", "meningkat", "menurun", "boleh", "berhasil", "gagal"
}
TEMPORAL_WORDS = {
    "senin","selasa","rabu","kamis","jumat","sabtu","minggu",
    "januari","februari","maret","april","mei","juni","juli","agustus",
    "september","oktober","november","desember","kemarin","hari","pekan","bulan","tahun"
}


def clean_ws(text):
    text = unicodedata.normalize("NFKC", str(text)).replace("\xa0", " ")
    return re.sub(r"\s+", " ", text).strip()


def clean_title_for_model(text):
    return SOURCE_SUFFIX_RE.sub("", clean_ws(text)).strip()


def normalize_key(text):
    return " ".join(TOKEN_RE.findall(clean_ws(text).lower()))


def tokens(text):
    return TOKEN_RE.findall(normalize_key(text))


def canonical_number(x):
    return re.sub(r"\D", "", str(x))


def raw_numbers(text):
    return [x for x in NUM_RAW_RE.findall(clean_ws(text)) if canonical_number(x)]

for df in (train, test):
    df["title_model"] = df.title.map(clean_title_for_model)
    df["title_key"] = df.title_model.map(normalize_key)
    df["content_key"] = df.content.map(normalize_key)
    df["pair_key"] = df.title_key + "|||" + df.content_key
    df["content_words"] = df.content.map(lambda x: len(clean_ws(x).split()))

# IDF is label-free and fit on train only.
doc_freq = Counter()
N_DOCS = len(train)
for t, c in tqdm(zip(train.title_model, train.content), total=N_DOCS, desc="IDF corpus"):
    doc_freq.update(set(tokens(t)) | set(tokens(c)))
idf = {tok: math.log((N_DOCS + 1) / (df + 1)) + 1.0 for tok, df in doc_freq.items()}
IDF_FLOOR = math.log((N_DOCS + 1) / 2) + 1.0
HIGH_IDF_THRESHOLD = float(np.quantile(np.array(list(idf.values()), dtype=float), HIGH_IDF_QUANTILE))


def tok_idf(tok):
    return idf.get(tok, IDF_FLOOR)


def title_window_score(title_toks, window_text):
    tset = set(title_toks)
    if not tset:
        return 0.0
    wset = set(tokens(window_text))
    denom = sum(tok_idf(t) for t in tset) + 1e-9
    return sum(tok_idf(t) for t in tset if t in wset) / denom


def make_windows(content, window_words, stride):
    words = clean_ws(content).split()
    if not words:
        return [(0, "")]
    if len(words) <= window_words:
        return [(0, " ".join(words))]
    starts = list(range(0, max(1, len(words) - window_words + 1), stride))
    last = max(0, len(words) - window_words)
    if starts[-1] != last:
        starts.append(last)
    return [(s, " ".join(words[s:s + window_words])) for s in starts]


def global_evidence(title, content):
    tt = tokens(title)
    windows = make_windows(content, GLOBAL_WINDOW_WORDS, GLOBAL_WINDOW_STRIDE)
    scored = sorted(
        [(title_window_score(tt, text), i) for i, (_, text) in enumerate(windows)],
        reverse=True,
    )
    selected = [0]
    for _, i in scored:
        if i not in selected:
            selected.append(i)
        if len(selected) >= GLOBAL_TOP_WINDOWS:
            break
    # V2 keeps the lead first, then puts the strongest retrieved spans first rather than sorting by article position.
    lead = [i for i in selected if i == 0]
    rest = [i for _, i in scored if i in selected and i != 0]
    order = lead + rest
    return " \n ".join(windows[i][1] for i in order)


def local_evidence_views(title, content, n_views=N_LOCAL_VIEWS):
    tt = tokens(title)
    windows = make_windows(content, LOCAL_WINDOW_WORDS, LOCAL_WINDOW_STRIDE)
    scores = [title_window_score(tt, text) for _, text in windows]
    selected = [0]
    # Prefer high-scoring windows while avoiding immediately adjacent near-duplicates when possible.
    for i in np.argsort(-np.asarray(scores)):
        i = int(i)
        if i in selected:
            continue
        start_i = windows[i][0]
        too_close = any(abs(start_i - windows[j][0]) < max(1, LOCAL_WINDOW_STRIDE) for j in selected)
        if too_close and len(windows) > n_views:
            continue
        selected.append(i)
        if len(selected) >= n_views:
            break
    if len(selected) < n_views:
        for i in np.argsort(-np.asarray(scores)):
            i = int(i)
            if i not in selected:
                selected.append(i)
            if len(selected) >= n_views:
                break

    views, valid = [], []
    for i in selected[:n_views]:
        views.append(windows[i][1])
        valid.append(True)
    while len(views) < n_views:
        views.append(views[-1] if views else "")
        valid.append(False)
    return views, valid, scores[0] if scores else 0.0, max(scores) if scores else 0.0


def add_views(df):
    out = df.copy()
    globals_, locals_, masks_, lead_scores_, best_scores_ = [], [], [], [], []
    for t, c in tqdm(zip(out.title_model, out.content), total=len(out), desc="Global/local evidence"):
        globals_.append(global_evidence(t, c))
        lv, lm, lead_s, best_s = local_evidence_views(t, c)
        locals_.append(lv); masks_.append(lm)
        lead_scores_.append(lead_s); best_scores_.append(best_s)
    out["global_evidence"] = globals_
    for k in range(N_LOCAL_VIEWS):
        out[f"local_{k}"] = [x[k] for x in locals_]
        out[f"local_valid_{k}"] = [bool(x[k]) for x in masks_]
    out["lead_relevance"] = lead_scores_
    out["best_local_relevance"] = best_scores_
    out["retrieval_gain"] = out.best_local_relevance - out.lead_relevance
    return out

train = add_views(train)
test = add_views(test)
print(train[["title_model", "global_evidence", "local_0", "local_1"]].head(2))

IDF corpus:   0%|          | 0/14397 [00:00<?, ?it/s]

Global/local evidence:   0%|          | 0/14397 [00:00<?, ?it/s]

Global/local evidence:   0%|          | 0/3603 [00:00<?, ?it/s]

                                                                        title_model  \
0                      Tak Lagi Jadi Jubir Yurianto Tak Umumkan Update Harian Covid   
1  Aturan Swab PCR Bagi Penumpang Pesawat ke Kalbar Diperpanjang hingga 28 Februari   

                                                                                                                                                                       global_evidence  \
0  Ada perubahan besar besaran dalam pemaparan data harian perkembangan virus corona Covid 19 di Indonesia Selasa 21 7 Selain tak lagi diumumkan Dirjen P2P Kemenkes Achmad Yuriant...   
1  Aturan Swab PCR Bagi Penumpang Pesawat ke Kalbar Diperpanjang hingga 28 FebruariHi Pontianak Pemerintah Provinsi Kalimantan Barat memperpanjang kebijakan persyaratan kepada pel...   

                                                                                                                                                                               loca

In [5]:
# Compact static features used only where they match the V1 error diagnosis.
def static_features(df):
    rows = []
    for r in tqdm(df.itertuples(), total=len(df), desc="Static features"):
        tt = tokens(r.title_model)
        tset = set(tt)
        content_set = set(tokens(r.content))
        idf_denom = sum(tok_idf(t) for t in tset) + 1e-9
        full_idf_cov = sum(tok_idf(t) for t in tset if t in content_set) / idf_denom
        rows.append({
            "title_words": len(tt),
            "content_words": int(r.content_words),
            "log_content_words": math.log1p(int(r.content_words)),
            "retrieval_gain": float(r.retrieval_gain),
            "full_idf_cov": float(full_idf_cov),
            "title_has_negation": int(any(t in NEGATIONS for t in tt)),
            "content_quality_short": int(int(r.content_words) < MIN_GRAPH_CONTENT_WORDS),
        })
    return pd.DataFrame(rows, index=df.index)

train_static = static_features(train)
test_static = static_features(test)
display(train_static.describe().T[["mean", "std", "min", "max"]])

Static features:   0%|          | 0/14397 [00:00<?, ?it/s]

Static features:   0%|          | 0/3603 [00:00<?, ?it/s]

,mean,std,min,max
title_words,9.948461,1.859166,3.000000,22.000000
content_words,309.950615,169.578057,3.000000,4238.000000
log_content_words,5.640710,0.438072,1.386294,8.352083
retrieval_gain,0.021889,0.075904,0.000000,1.000000
full_idf_cov,0.759067,0.164487,0.000000,1.000000
title_has_negation,0.114468,0.318390,0.000000,1.000000
content_quality_short,0.000625,0.024996,0.000000,1.000000


## Strict connected-component CV

Rows sharing a normalized title **or** normalized content stay in the same fold. This remains the semantic-generalization score used for model selection. Graph exploitation is evaluated separately with leave-one-out diagnostics.

In [6]:
class UnionFind:
    def __init__(self, n):
        self.p = list(range(n)); self.r = [0] * n
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb:
            return
        if self.r[ra] < self.r[rb]:
            ra, rb = rb, ra
        self.p[rb] = ra
        if self.r[ra] == self.r[rb]:
            self.r[ra] += 1


def build_component_groups(df):
    uf = UnionFind(len(df))
    for col in ["title_key", "content_key"]:
        first = {}
        for i, key in enumerate(df[col].tolist()):
            if key in first:
                uf.union(i, first[key])
            else:
                first[key] = i
    roots = [uf.find(i) for i in range(len(df))]
    mapping = {r: j for j, r in enumerate(dict.fromkeys(roots))}
    return np.array([mapping[r] for r in roots], dtype=np.int32)

groups = build_component_groups(train)
train["cv_group"] = groups
cv = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
splits = list(cv.split(train, train.label.values, groups=groups))
print("Connected components:", train.cv_group.nunique())
print("Rows in non-singleton components:", int(train.groupby("cv_group").size().loc[lambda s: s > 1].sum()))
for fold, (tr_idx, va_idx) in enumerate(splits):
    assert set(groups[tr_idx]).isdisjoint(set(groups[va_idx]))
    print(f"fold={fold} train={len(tr_idx):5d} valid={len(va_idx):5d} valid_neg={(train.iloc[va_idx].label==0).mean():.3f}")

Connected components: 11525
Rows in non-singleton components: 5419
fold=0 train= 9599 valid= 4798 valid_neg=0.099
fold=1 train= 9590 valid= 4807 valid_neg=0.096
fold=2 train= 9605 valid= 4792 valid_neg=0.104


## V2 graph audit

The V1 content rule was highly accurate but its mistakes came from content that legitimately had multiple positive headlines. V2 therefore only hard-forces a content mismatch when the reference content has **exactly one canonical positive title** and the content is not a short/placeholder body.

The old raw same-title rule remains too risky to hard-force. V2 keeps it only as a **confidence-gated** rule learned from OOF model probabilities later in the notebook.

In [7]:
def graph_loo_audit(df):
    pair_counts = defaultdict(Counter)
    content_pos = defaultdict(Counter)
    title_pos = defaultdict(Counter)
    for r in df.itertuples():
        pair_counts[r.pair_key][int(r.label)] += 1
        if int(r.label) == 1:
            content_pos[r.content_key][r.title_key] += 1
            title_pos[r.title_key][r.content_key] += 1

    records = []
    for r in df.itertuples():
        y = int(r.label)
        pc = pair_counts[r.pair_key].copy(); pc[y] -= 1
        if pc[y] <= 0: pc.pop(y, None)
        if sum(pc.values()) > 0 and len(pc) == 1:
            records.append((r.Index, "exact_pair", next(iter(pc.keys())), y))
            continue

        # Guard with FULL reference cardinality. This is the V2 fix for multi-headline content:
        # if a content has >1 known positive headline anywhere in train, never treat it as canonical-unique.
        full_positive_title_count = len([k for k, v in content_pos[r.content_key].items() if v > 0])
        cp = content_pos[r.content_key].copy()
        if y == 1:
            cp[r.title_key] -= 1
            if cp[r.title_key] <= 0: cp.pop(r.title_key, None)
        positive_titles = {k for k, v in cp.items() if v > 0}
        if (
            full_positive_title_count == 1
            and len(positive_titles) == 1
            and r.title_key not in positive_titles
            and int(r.content_words) >= MIN_GRAPH_CONTENT_WORDS
        ):
            records.append((r.Index, "unique_content_different_title", 0, y))
            continue

        tp = title_pos[r.title_key].copy()
        if y == 1:
            tp[r.content_key] -= 1
            if tp[r.content_key] <= 0: tp.pop(r.content_key, None)
        positive_contents = {k for k, v in tp.items() if v > 0}
        if positive_contents and r.content_key not in positive_contents and int(r.content_words) >= MIN_GRAPH_CONTENT_WORDS:
            records.append((r.Index, "title_candidate", -1, y))

    return pd.DataFrame(records, columns=["index", "rule", "pred", "y"]).set_index("index")

graph_audit = graph_loo_audit(train)
for rule, g in graph_audit.groupby("rule"):
    if rule == "title_candidate":
        print(rule, "coverage=", len(g), "(gated later)")
    else:
        print(rule, "coverage=", len(g), "accuracy=", round(float((g.pred == g.y).mean()), 6))

unique_rows = graph_audit[graph_audit.rule == "unique_content_different_title"]
UNIQUE_CONTENT_RULE_ENABLED = (
    len(unique_rows) > 0 and float((unique_rows.pred == unique_rows.y).mean()) >= UNIQUE_CONTENT_RULE_MIN_PRECISION
)
print("unique-content rule enabled:", UNIQUE_CONTENT_RULE_ENABLED)

pair_label_sets = train.groupby("pair_key").label.agg(lambda s: set(map(int, s)))
assert int(pair_label_sets.map(len).gt(1).sum()) == 0, "Conflicting exact-pair labels detected"
print("test exact pairs seen in train:", int(test.pair_key.isin(pair_label_sets.index).sum()))

exact_pair coverage= 3620 accuracy= 1.0
title_candidate coverage= 40 (gated later)
unique_content_different_title coverage= 909 accuracy= 1.0
unique-content rule enabled: True
test exact pairs seen in train: 927


## Hard real pairs + conservative counterfactual curriculum

The highest-value relation supervision already exists inside the dataset: the same article paired with a correct and an incorrect headline. V2 chooses the **lexically hardest opposite-label title** inside each train fold.

Synthetic counterfactuals are deliberately capped and low-weight. They are generated only from positive examples and only when a conservative mutation can be made without inserting a replacement already present in the article. This is auxiliary supervision, never a replacement for real negatives.

In [8]:
def title_similarity(a, b):
    ta, tb = tokens(a), tokens(b)
    sa, sb = set(ta), set(tb)
    jac = len(sa & sb) / max(1, len(sa | sb))
    seq = difflib.SequenceMatcher(None, ta, tb).ratio()
    return 0.6 * seq + 0.4 * jac


def best_local_for_title(title, content):
    views, valid, _, _ = local_evidence_views(title, content)
    scores = [title_window_score(tokens(title), v) if ok else -1 for v, ok in zip(views, valid)]
    return views[int(np.argmax(scores))]


def build_hard_pair_map(train_fold):
    grouped = defaultdict(lambda: {0: [], 1: []})
    for idx, r in train_fold.iterrows():
        grouped[r.content_key][int(r.label)].append((int(idx), r.title_model, r.content))
    out = {}
    for _, g in grouped.items():
        if not g[0] or not g[1]:
            continue
        for y in (0, 1):
            opposite = g[1-y]
            for idx, title, content in g[y]:
                _, opp_title, _ = max(opposite, key=lambda z: title_similarity(title, z[1]))
                out[idx] = {
                    "title": opp_title,
                    "premise": best_local_for_title(opp_title, content),
                }
    return out

POLARITY_SWAP = {
    "belum": "sudah", "sudah": "belum", "positif": "negatif", "negatif": "positif",
    "naik": "turun", "turun": "naik", "meningkat": "menurun", "menurun": "meningkat",
    "boleh": "dilarang", "dilarang": "boleh", "berhasil": "gagal", "gagal": "berhasil",
}
ENTITY_RE = re.compile(r"\b(?:[A-Z][a-z]+|[A-Z]{2,})(?:\s+(?:[A-Z][a-z]+|[A-Z]{2,})){0,2}\b")
ENTITY_STOP = {
    "Covid", "Corona", "COVID", "Indonesia", "Senin", "Selasa", "Rabu", "Kamis", "Jumat", "Sabtu", "Minggu"
}


def deterministic_unit(key):
    h = hashlib.sha1((str(SEED) + "|" + str(key)).encode()).hexdigest()
    return int(h[:8], 16) / 0xFFFFFFFF


def build_entity_pool(train_fold):
    pool = Counter()
    for t in train_fold.loc[train_fold.label == 1, "title_model"]:
        for e in ENTITY_RE.findall(t):
            e = clean_ws(e)
            if e not in ENTITY_STOP and 1 <= len(e.split()) <= 3:
                pool[e] += 1
    return [e for e, _ in pool.most_common(500)]


def replace_word_case_insensitive(text, old, new):
    return re.sub(rf"\b{re.escape(old)}\b", new, text, count=1, flags=re.I)


def make_synthetic_title(row, number_pool, entity_pool):
    if int(row.label) != 1 or deterministic_unit(row.id) >= SYNTHETIC_RATE:
        return None, None
    title = clean_title_for_model(row.title_model)
    content_key = normalize_key(row.content)

    # 1) Polarity mutation: preferred because it directly targets a weak V1 slice.
    tt = tokens(title)
    for old in tt:
        if old in POLARITY_SWAP:
            new = POLARITY_SWAP[old]
            if new not in set(tokens(row.content)):
                mutated = replace_word_case_insensitive(title, old, new)
                if mutated != title:
                    return mutated, "polarity"

    # 2) Numeric mutation with a replacement number not present in the article.
    nums = raw_numbers(title)
    content_nums = {canonical_number(x) for x in raw_numbers(row.content)}
    if nums:
        old = nums[0]
        old_c = canonical_number(old)
        candidates = [n for n in number_pool if canonical_number(n) not in content_nums and canonical_number(n) != old_c]
        if candidates:
            j = int(deterministic_unit(f"{row.id}|num") * len(candidates)) % len(candidates)
            new = candidates[j]
            return title.replace(old, new, 1), "number"

    # 3) Entity/location/role-like mutation. Require the original span to be supported and replacement absent.
    entities = [e for e in ENTITY_RE.findall(title) if e not in ENTITY_STOP and normalize_key(e) in content_key]
    if entities:
        old = max(entities, key=len)
        candidates = [e for e in entity_pool if normalize_key(e) not in content_key and e != old and len(e.split()) == len(old.split())]
        if candidates:
            j = int(deterministic_unit(f"{row.id}|ent") * len(candidates)) % len(candidates)
            new = candidates[j]
            return title.replace(old, new, 1), "entity"

    return None, None


def build_synthetic_map(train_fold):
    number_pool = []
    for t in train_fold.loc[train_fold.label == 1, "title_model"]:
        number_pool.extend(raw_numbers(t))
    # keep a stable, diverse pool
    number_pool = list(dict.fromkeys(number_pool))
    entity_pool = build_entity_pool(train_fold)
    out = {}
    types = Counter()
    if not USE_SYNTHETIC_COUNTERFACTUALS:
        return out, types
    for idx, r in train_fold.iterrows():
        title, typ = make_synthetic_title(r, number_pool, entity_pool)
        if title is not None:
            out[int(idx)] = {
                "title": title,
                "premise": best_local_for_title(r.title_model, r.content),
                "type": typ,
            }
            types[typ] += 1
    return out, types

## NLI model, AMP safety, multi-view batching, and exact threshold search

The checkpoint declares FP16 dtype upstream, so V2 keeps the V1.1.1 fix: **load FP32 master weights explicitly** and use FP16 only inside CUDA autocast. This prevents the `GradScaler.unscale_()` crash seen in the first optimized V1 attempt.

In [9]:
def resolve_model_source():
    if LOCAL_MODEL_PATH:
        return LOCAL_MODEL_PATH
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for cfg in kaggle_input.rglob("config.json"):
            parent = cfg.parent
            name = str(parent).lower()
            if "mdeberta" in name and (parent / "tokenizer_config.json").exists():
                print("Using local Kaggle model:", parent)
                return str(parent)
    return MODEL_NAME


def ensure_fp32_master_weights(model, verbose=True):
    bad = [p for p in model.parameters() if p.is_floating_point() and p.dtype != torch.float32]
    if bad:
        if verbose:
            print(f"Casting {len(bad)} non-FP32 tensors to FP32 master weights")
        model.float()
    remaining = {str(p.dtype) for p in model.parameters() if p.is_floating_point() and p.dtype != torch.float32}
    if remaining:
        raise RuntimeError(f"FP32 master-weight invariant failed: {sorted(remaining)}")
    if verbose:
        print("Master parameter dtypes:", sorted({str(p.dtype) for p in model.parameters() if p.is_floating_point()}))
    return model


def load_tokenizer_and_model():
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    source = resolve_model_source()
    tokenizer = AutoTokenizer.from_pretrained(source, use_fast=True)
    try:
        model = AutoModelForSequenceClassification.from_pretrained(source, dtype=torch.float32)
    except TypeError:
        model = AutoModelForSequenceClassification.from_pretrained(source, torch_dtype=torch.float32)
    model = ensure_fp32_master_weights(model)
    if model.config.num_labels != 3:
        raise ValueError(f"Expected a 3-label NLI head, got {model.config.num_labels}")
    model.config.use_cache = False
    if USE_GRAD_CHECKPOINTING and hasattr(model, "gradient_checkpointing_enable"):
        model.gradient_checkpointing_enable()
        print("Gradient checkpointing: ON")
    else:
        print("Gradient checkpointing: OFF")
    print("NLI id2label:", model.config.id2label)
    return tokenizer, model


def nli_indices(model):
    cfg = model.module.config if isinstance(model, nn.DataParallel) else model.config
    id2label = {int(k): str(v).lower() for k, v in cfg.id2label.items()}
    def find(name, fallback):
        for k, v in id2label.items():
            if name in v:
                return int(k)
        return fallback
    return find("entail", 0), find("neutral", 1), find("contrad", 2)


def unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model


def move_encoding(enc):
    return {k: v.to(DEVICE, non_blocking=True) for k, v in enc.items()}


def amp_context():
    return torch.autocast("cuda", dtype=torch.float16) if AMP_ENABLED else nullcontext()


def make_grad_scaler():
    try:
        return torch.amp.GradScaler("cuda", enabled=AMP_ENABLED)
    except (AttributeError, TypeError):
        return torch.cuda.amp.GradScaler(enabled=AMP_ENABLED)


def best_threshold(y_true, probs):
    y = np.asarray(y_true, dtype=np.int8)
    p = np.asarray(probs, dtype=np.float64)
    if len(y) == 0 or y.shape != p.shape or not np.isfinite(p).all():
        raise ValueError("best_threshold received invalid arrays")
    order = np.argsort(-p, kind="mergesort")
    ps, ys = p[order], y[order]
    pos_total = float(ys.sum()); neg_total = float(len(ys) - ys.sum())
    tp_cum = np.cumsum(ys, dtype=float); fp_cum = np.cumsum(1 - ys, dtype=float)
    boundaries = np.flatnonzero(np.r_[ps[1:] != ps[:-1], True])
    tp = tp_cum[boundaries]; fp = fp_cum[boundaries]
    fn = pos_total - tp; tn = neg_total - fp
    f1_pos = 2*tp / np.maximum(1.0, 2*tp + fp + fn)
    f1_neg = 2*tn / np.maximum(1.0, 2*tn + fp + fn)
    scores = 0.5*(f1_pos + f1_neg)
    thresholds = ps[boundaries]
    i = int(np.argmax(scores))
    return float(thresholds[i]), float(scores[i])


def probability_diagnostics(y_true, probs, prefix=""):
    y = np.asarray(y_true, dtype=np.int8); p = np.asarray(probs, dtype=float)
    t, macro = best_threshold(y, p)
    pred = (p >= t).astype(np.int8)
    f1s = f1_score(y, pred, labels=[0,1], average=None, zero_division=0)
    cm = confusion_matrix(y, pred, labels=[0,1])
    q = np.quantile(p, [0, .01, .10, .50, .90, .99, 1])
    print(f"{prefix} macro_f1={macro:.6f} t={t:.6g} F1_0={f1s[0]:.4f} F1_1={f1s[1]:.4f}")
    print(f"{prefix} p[min,p01,p10,p50,p90,p99,max]=" + ", ".join(f"{x:.5f}" for x in q))
    print(f"{prefix} confusion_matrix={cm.tolist()}")
    return {"threshold": t, "macro_f1": macro, "f1_0": float(f1s[0]), "f1_1": float(f1s[1])}


def binary_partial_nli_loss(probs, y, entail_idx, neg_weight=NEG_CLASS_WEIGHT):
    pe = probs[:, entail_idx].clamp(1e-6, 1-1e-6)
    target_prob = torch.where(y > 0.5, pe, 1.0 - pe)
    loss = -torch.log(target_prob)
    weights = torch.where(y < 0.5, torch.full_like(y, neg_weight), torch.ones_like(y))
    return (loss * weights).mean()


def masked_softmax_pool(values, mask, alpha=MIL_POOL_ALPHA):
    # values/mask: [B,K]
    logits = alpha * values
    logits = logits.masked_fill(~mask, -1e4)
    weights = torch.softmax(logits, dim=1)
    return (weights * values).sum(dim=1)


def safe_logit(p):
    p = p.clamp(1e-5, 1-1e-5)
    return torch.log(p) - torch.log1p(-p)

# Regression guard for the previous AMP failure mode.
_probe = nn.Linear(2, 2).half()
ensure_fp32_master_weights(_probe, verbose=False)
assert all(p.dtype == torch.float32 for p in _probe.parameters())
del _probe

In [10]:
class MultiViewDataset(Dataset):
    def __init__(self, df, pair_map=None, synth_map=None):
        self.df = df.reset_index(drop=False).rename(columns={"index": "orig_index"})
        self.pair_map = pair_map or {}
        self.synth_map = synth_map or {}
    def __len__(self):
        return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        idx = int(r.orig_index)
        locals_ = [r[f"local_{k}"] for k in range(N_LOCAL_VIEWS)]
        mask = [bool(r[f"local_valid_{k}"]) for k in range(N_LOCAL_VIEWS)]
        return {
            "orig_index": idx,
            "title": r.title_model,
            "global": r.global_evidence,
            "locals": locals_,
            "local_mask": mask,
            "label": int(r.label) if "label" in self.df.columns else -1,
            "pair": self.pair_map.get(idx),
            "synth": self.synth_map.get(idx),
        }


def make_multiview_collator(tokenizer, training=False):
    def collate(batch):
        B = len(batch)
        global_enc = tokenizer(
            [x["global"] for x in batch], [x["title"] for x in batch],
            padding=True, truncation="only_first", max_length=GLOBAL_MAX_LENGTH,
            pad_to_multiple_of=8, return_tensors="pt"
        )

        local_premises, local_titles = [], []
        for x in batch:
            local_premises.extend(x["locals"])
            local_titles.extend([x["title"]] * N_LOCAL_VIEWS)
        main_local_count = len(local_premises)

        pair_positions = []
        synth_positions = []
        if training:
            pair_positions = [i for i, x in enumerate(batch) if x["pair"] is not None][:MAX_PAIRWISE_PER_BATCH]
            synth_positions = [i for i, x in enumerate(batch) if x["synth"] is not None][:MAX_SYNTHETIC_PER_BATCH]
            for i in pair_positions:
                local_premises.append(batch[i]["pair"]["premise"])
                local_titles.append(batch[i]["pair"]["title"])
            for i in synth_positions:
                local_premises.append(batch[i]["synth"]["premise"])
                local_titles.append(batch[i]["synth"]["title"])

        local_enc = tokenizer(
            local_premises, local_titles,
            padding=True, truncation="only_first", max_length=LOCAL_MAX_LENGTH,
            pad_to_multiple_of=8, return_tensors="pt"
        )
        return {
            "global_enc": global_enc,
            "local_enc": local_enc,
            "labels": torch.tensor([x["label"] for x in batch], dtype=torch.float32),
            "orig_index": torch.tensor([x["orig_index"] for x in batch], dtype=torch.long),
            "local_mask": torch.tensor([x["local_mask"] for x in batch], dtype=torch.bool),
            "main_local_count": main_local_count,
            "pair_positions": torch.tensor(pair_positions, dtype=torch.long),
            "synth_positions": torch.tensor(synth_positions, dtype=torch.long),
            "synth_types": [batch[i]["synth"]["type"] for i in synth_positions],
        }
    return collate


def loader_kwargs():
    kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
    if NUM_WORKERS > 0:
        kw.update(persistent_workers=True, prefetch_factor=2)
    return kw

## Multi-view prediction features

The NLI head remains three-way (`entailment / neutral / contradiction`). V2 never collapses those states before the meta layer. The semantic support score is a calibrated combination of global entailment and local MIL entailment; the complete NLI state is retained as features.

In [11]:
@torch.no_grad()
def predict_multiview(model, loader, desc="predict"):
    model.eval()
    entail_idx, neutral_idx, contradiction_idx = nli_indices(model)
    rows = []
    for batch in tqdm(loader, desc=desc):
        genc = move_encoding(batch["global_enc"])
        lenc = move_encoding(batch["local_enc"])
        mask = batch["local_mask"].to(DEVICE, non_blocking=True)
        B = mask.shape[0]
        with amp_context():
            glogits = model(**genc, return_dict=False)[0]
            llogits = model(**lenc, return_dict=False)[0][:B * N_LOCAL_VIEWS]
            gp = torch.softmax(glogits, dim=-1)
            lp = torch.softmax(llogits, dim=-1).view(B, N_LOCAL_VIEWS, 3)
            local_e = lp[:, :, entail_idx]
            local_pool_e = masked_softmax_pool(local_e, mask)
            semantic_raw = GLOBAL_LOSS_WEIGHT * gp[:, entail_idx] + LOCAL_LOSS_WEIGHT * local_pool_e
            masked_e = local_e.masked_fill(~mask, -1.0)
            best_idx = masked_e.argmax(dim=1)
            arange = torch.arange(B, device=DEVICE)
            best_probs = lp[arange, best_idx]
            local_max_c = lp[:, :, contradiction_idx].masked_fill(~mask, -1.0).max(dim=1).values
            local_max_n = lp[:, :, neutral_idx].masked_fill(~mask, -1.0).max(dim=1).values
            topk = torch.topk(masked_e, k=min(2, N_LOCAL_VIEWS), dim=1).values.clamp_min(0)
            topk_denom = mask.sum(dim=1).clamp(min=1, max=min(2, N_LOCAL_VIEWS)).to(topk.dtype)
            local_top2_e = topk.sum(dim=1) / topk_denom

        for j in range(B):
            rows.append({
                "orig_index": int(batch["orig_index"][j]),
                "semantic_raw": float(semantic_raw[j]),
                "global_e": float(gp[j, entail_idx]),
                "global_n": float(gp[j, neutral_idx]),
                "global_c": float(gp[j, contradiction_idx]),
                "local_pool_e": float(local_pool_e[j]),
                "local_max_e": float(masked_e[j].max()),
                "local_top2_e": float(local_top2_e[j]),
                "local_max_n": float(local_max_n[j]),
                "local_max_c": float(local_max_c[j]),
                "best_local_e": float(best_probs[j, entail_idx]),
                "best_local_n": float(best_probs[j, neutral_idx]),
                "best_local_c": float(best_probs[j, contradiction_idx]),
                "global_local_gap": float(torch.abs(gp[j, entail_idx] - local_pool_e[j])),
                "best_local_idx": int(best_idx[j]),
            })
    return pd.DataFrame(rows).set_index("orig_index").sort_index()


def fit_platt(y, p):
    x = np.log(np.clip(np.asarray(p, dtype=float), 1e-5, 1-1e-5) / np.clip(1-np.asarray(p, dtype=float), 1e-5, 1))[:, None]
    model = LogisticRegression(C=10.0, max_iter=1000, solver="liblinear", random_state=SEED)
    model.fit(x, np.asarray(y, dtype=int))
    return model


def apply_platt(model, p):
    p = np.asarray(p, dtype=float)
    x = np.log(np.clip(p, 1e-5, 1-1e-5) / np.clip(1-p, 1e-5, 1))[:, None]
    return model.predict_proba(x)[:, 1]


def crossfit_calibrate(y, p, seed):
    y = np.asarray(y, dtype=int); p = np.asarray(p, dtype=float)
    skf = StratifiedKFold(n_splits=2, shuffle=True, random_state=seed)
    out = np.full(len(y), np.nan, dtype=float)
    for tr, va in skf.split(np.zeros(len(y)), y):
        m = fit_platt(y[tr], p[tr])
        out[va] = apply_platt(m, p[va])
    assert np.isfinite(out).all()
    return out, fit_platt(y, p)

## Fold training

Memory optimization is intentional:

- the global forward/backward is completed first,
- local MIL + pairwise + synthetic auxiliary losses are then forwarded/backwarded,
- gradients accumulate into the same optimizer step.

This preserves the V2 objective while avoiding simultaneous retention of global and local activation graphs on 16 GB T4 cards.

In [12]:
def train_one_fold(fold, tr_idx, va_idx, test_df):
    from transformers import get_cosine_schedule_with_warmup

    tr_df = train.iloc[tr_idx].copy()
    va_df = train.iloc[va_idx].copy()
    pair_map = build_hard_pair_map(tr_df)
    synth_map, synth_types = build_synthetic_map(tr_df)
    print(f"Fold {fold}: hard-pair rows={len(pair_map)} | synthetic rows={len(synth_map)} | types={dict(synth_types)}")

    tokenizer, base_model = load_tokenizer_and_model()
    base_model.to(DEVICE)
    ensure_fp32_master_weights(base_model)
    model = nn.DataParallel(base_model) if GPU_COUNT > 1 else base_model
    if GPU_COUNT > 1:
        print(f"Using DataParallel on {GPU_COUNT} GPUs")
    entail_idx, neutral_idx, contradiction_idx = nli_indices(model)

    train_collate = make_multiview_collator(tokenizer, training=True)
    eval_collate = make_multiview_collator(tokenizer, training=False)
    fold_generator = torch.Generator()
    fold_generator.manual_seed(SEED + fold)
    tr_loader = DataLoader(MultiViewDataset(tr_df, pair_map, synth_map), batch_size=BATCH_SIZE, shuffle=True,
                           collate_fn=train_collate, generator=fold_generator, **loader_kwargs())
    va_loader = DataLoader(MultiViewDataset(va_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
                           collate_fn=eval_collate, **loader_kwargs())
    te_loader = DataLoader(MultiViewDataset(test_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
                           collate_fn=eval_collate, **loader_kwargs())

    # Cheap zero-shot sanity check before training.
    diag_df = va_df.sample(n=min(384, len(va_df)), random_state=SEED + fold)
    diag_loader = DataLoader(MultiViewDataset(diag_df), batch_size=EVAL_BATCH_SIZE, shuffle=False,
                             collate_fn=eval_collate, **loader_kwargs())
    z = predict_multiview(model, diag_loader, desc=f"fold {fold} zero-shot")
    z_aligned = z.loc[diag_df.index, "semantic_raw"].to_numpy()
    probability_diagnostics(diag_df.label.values, z_aligned, prefix=f"fold={fold} zero-shot")
    del diag_loader, z

    dtypes = sorted({str(p.dtype) for p in unwrap_model(model).parameters() if p.is_floating_point()})
    if dtypes != ["torch.float32"]:
        raise RuntimeError(f"Refusing optimizer creation with master dtypes={dtypes}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = make_grad_scaler()
    update_steps_per_epoch = math.ceil(len(tr_loader) / GRAD_ACCUM)
    total_steps = update_steps_per_epoch * EPOCHS
    scheduler = get_cosine_schedule_with_warmup(optimizer, int(total_steps * WARMUP_RATIO), total_steps)

    best_f1 = -1.0; best_state = None; best_threshold_value = 0.5
    history = []
    optimizer.zero_grad(set_to_none=True)

    for epoch in range(EPOCHS):
        start = time.perf_counter()
        model.train()
        running_total, running_global, running_local = [], [], []
        pair_seen = synth_seen = 0
        synth_type_seen = Counter()
        pbar = tqdm(tr_loader, desc=f"fold {fold} epoch {epoch+1}/{EPOCHS}")

        for step, batch in enumerate(pbar):
            y = batch["labels"].to(DEVICE, non_blocking=True)
            mask = batch["local_mask"].to(DEVICE, non_blocking=True)
            B = len(y)

            # Global objective first. Backward immediately so its activations are released before local forward.
            genc = move_encoding(batch["global_enc"])
            with amp_context():
                glogits = model(**genc, return_dict=False)[0]
                gp = torch.softmax(glogits, dim=-1)
                global_loss = binary_partial_nli_loss(gp, y, entail_idx)
                global_scaled = GLOBAL_LOSS_WEIGHT * global_loss
            scaler.scale(global_scaled / GRAD_ACCUM).backward()
            del genc, glogits, gp, global_scaled

            # Local MIL main views + capped pairwise/synthetic auxiliary sequences in one forward.
            lenc = move_encoding(batch["local_enc"])
            pair_positions = batch["pair_positions"].to(DEVICE, non_blocking=True)
            synth_positions = batch["synth_positions"].to(DEVICE, non_blocking=True)
            pair_seen += int(pair_positions.numel()); synth_seen += int(synth_positions.numel())
            synth_type_seen.update(batch["synth_types"])

            with amp_context():
                all_llogits = model(**lenc, return_dict=False)[0]
                main_count = B * N_LOCAL_VIEWS
                main_logits = all_llogits[:main_count].view(B, N_LOCAL_VIEWS, 3)
                main_probs = torch.softmax(main_logits, dim=-1)
                local_e = main_probs[:, :, entail_idx]
                local_support = masked_softmax_pool(local_e, mask)
                local_target = torch.where(y > 0.5, local_support, 1.0 - local_support).clamp(1e-6, 1.0)
                local_item_loss = -torch.log(local_target)
                local_weights = torch.where(y < 0.5, torch.full_like(y, NEG_CLASS_WEIGHT), torch.ones_like(y))
                local_loss = (local_item_loss * local_weights).mean()

                cursor = main_count
                rank_loss = torch.zeros((), device=DEVICE)
                if pair_positions.numel() > 0:
                    n_pair = int(pair_positions.numel())
                    pair_logits = all_llogits[cursor:cursor+n_pair]
                    cursor += n_pair
                    pair_p = torch.softmax(pair_logits, dim=-1)[:, entail_idx]
                    current_p = local_support[pair_positions]
                    current_y = y[pair_positions]
                    current_logit = safe_logit(current_p)
                    pair_logit = safe_logit(pair_p)
                    better = torch.where(current_y > 0.5, current_logit, pair_logit)
                    worse = torch.where(current_y > 0.5, pair_logit, current_logit)
                    rank_loss = F.relu(PAIRWISE_MARGIN - (better - worse)).mean()

                synth_loss = torch.zeros((), device=DEVICE)
                if synth_positions.numel() > 0:
                    n_synth = int(synth_positions.numel())
                    synth_logits = all_llogits[cursor:cursor+n_synth]
                    synth_target = torch.full((n_synth,), contradiction_idx, device=DEVICE, dtype=torch.long)
                    synth_loss = F.cross_entropy(synth_logits, synth_target)

                local_total = LOCAL_LOSS_WEIGHT * local_loss + PAIRWISE_LAMBDA * rank_loss + SYNTHETIC_LAMBDA * synth_loss
            scaler.scale(local_total / GRAD_ACCUM).backward()

            total_value = GLOBAL_LOSS_WEIGHT * float(global_loss.detach().float().cpu()) + float(local_total.detach().float().cpu())
            running_total.append(total_value)
            running_global.append(float(global_loss.detach().float().cpu()))
            running_local.append(float(local_loss.detach().float().cpu()))
            del lenc, all_llogits, main_logits, main_probs, local_e, local_support, local_total

            do_update = ((step + 1) % GRAD_ACCUM == 0) or (step + 1 == len(tr_loader))
            if do_update:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                old_scale = scaler.get_scale()
                scaler.step(optimizer); scaler.update()
                if scaler.get_scale() >= old_scale:
                    scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if step % 25 == 0:
                pbar.set_postfix(loss=f"{np.mean(running_total[-50:]):.4f}", scale=f"{scaler.get_scale():.0f}")

        va_pred = predict_multiview(model, va_loader, desc=f"fold {fold} validation")
        aligned = va_pred.loc[va_df.index, "semantic_raw"].to_numpy()
        diag = probability_diagnostics(va_df.label.values, aligned, prefix=f"fold={fold} epoch={epoch+1}")
        elapsed = time.perf_counter() - start
        diag.update({
            "fold": fold, "epoch": epoch+1,
            "train_loss": float(np.mean(running_total)),
            "global_loss": float(np.mean(running_global)),
            "local_loss": float(np.mean(running_local)),
            "pair_rows_used": int(pair_seen), "synthetic_rows_used": int(synth_seen),
            "synthetic_types": json.dumps(dict(synth_type_seen), sort_keys=True),
            "epoch_seconds": float(elapsed), "primary_rows_per_sec": float(len(tr_df)/max(1e-9, elapsed)),
            "amp_scale": float(scaler.get_scale()),
        })
        history.append(diag)
        print(f"fold={fold} epoch={epoch+1} time={elapsed/60:.2f} min rows/s={diag['primary_rows_per_sec']:.2f}")

        if diag["macro_f1"] > best_f1:
            best_f1 = diag["macro_f1"]
            best_threshold_value = diag["threshold"]
            best_state = {k: v.detach().cpu().clone() for k, v in unwrap_model(model).state_dict().items()}

    if best_state is None:
        raise RuntimeError("No checkpoint selected")
    unwrap_model(model).load_state_dict(best_state)

    val_features = predict_multiview(model, va_loader, desc=f"fold {fold} best validation")
    test_features = predict_multiview(model, te_loader, desc=f"fold {fold} test")

    # Within-fold cross-fitted calibration for validation; full validation fit for this fold's test scores.
    val_raw = val_features.loc[va_df.index, "semantic_raw"].to_numpy()
    val_cal, platt = crossfit_calibrate(va_df.label.values, val_raw, SEED + 100 + fold)
    val_features.loc[va_df.index, "semantic_cal"] = val_cal
    test_features["semantic_cal"] = apply_platt(platt, test_features["semantic_raw"].to_numpy())

    if SAVE_FOLD_MODELS:
        torch.save({
            "state_dict": best_state,
            "model_name": MODEL_NAME,
            "fold": fold,
            "best_macro_f1": best_f1,
            "best_threshold": best_threshold_value,
        }, OUTPUT_DIR / f"ifest_v2_fold{fold}.pt")

    del model, base_model, tokenizer, tr_loader, va_loader, te_loader, optimizer, scheduler, scaler, best_state
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return val_features, test_features, history

## Train three strict folds and collect OOF / test multi-view features

In [13]:
OOF_MODEL_COLS = [
    "semantic_raw", "semantic_cal", "global_e", "global_n", "global_c",
    "local_pool_e", "local_max_e", "local_top2_e", "local_max_n", "local_max_c",
    "best_local_e", "best_local_n", "best_local_c", "global_local_gap", "best_local_idx"
]
oof_model = pd.DataFrame(index=train.index, columns=OOF_MODEL_COLS, dtype=float)
test_fold_features = []
train_history = []

for fold, (tr_idx, va_idx) in enumerate(splits):
    vf, tf, hist = train_one_fold(fold, tr_idx, va_idx, test)
    for col in OOF_MODEL_COLS:
        oof_model.loc[vf.index, col] = vf[col].astype(float)
    tf = tf.copy(); tf["fold"] = fold
    test_fold_features.append(tf)
    train_history.extend(hist)

assert np.isfinite(oof_model.drop(columns=["best_local_idx"]).to_numpy(dtype=float)).all()
assert not oof_model.isna().any().any()

# Fold-calibrated support is the primary strict semantic diagnostic.
semantic_diag = probability_diagnostics(train.label.values, oof_model.semantic_cal.values, prefix="V2 calibrated semantic OOF")
semantic_t = semantic_diag["threshold"]
print(classification_report(train.label.values, (oof_model.semantic_cal.values >= semantic_t).astype(int), digits=4))

# Test: average each fold after fold-specific calibration. Recompute best local index from averaged local entailment is not possible
# from max-only features, so relational features below are averaged fold-by-fold using each fold's own selected best span.
test_model = pd.concat(test_fold_features).groupby(level=0)[[c for c in OOF_MODEL_COLS if c != "best_local_idx"]].mean()
assert test_model.index.tolist() == test.index.tolist()

Fold 0: hard-pair rows=1333 | synthetic rows=1115 | types={'number': 505, 'polarity': 189, 'entity': 421}


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/467 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/16.3M [00:00<?, ?B/s]

...

fold 2 validation:   0%|          | 0/400 [00:00<?, ?it/s]

fold=2 epoch=2 macro_f1=0.842094 t=0.310683 F1_0=0.7136 F1_1=0.9705
fold=2 epoch=2 p[min,p01,p10,p50,p90,p99,max]=0.00037, 0.00106, 0.46402, 0.98761, 0.99863, 0.99935, 0.99955
fold=2 epoch=2 confusion_matrix=[[319, 181], [75, 4217]]
fold=2 epoch=2 time=23.73 min rows/s=6.74


fold 2 best validation:   0%|          | 0/400 [00:00<?, ?it/s]

fold 2 test:   0%|          | 0/301 [00:00<?, ?it/s]

V2 calibrated semantic OOF macro_f1=0.828754 t=0.568159 F1_0=0.6886 F1_1=0.9689
V2 calibrated semantic OOF p[min,p01,p10,p50,p90,p99,max]=0.00458, 0.02325, 0.68391, 0.98251, 0.99625, 0.99791, 0.99880
V2 calibrated semantic OOF confusion_matrix=[[900, 537], [277, 12683]]
              precision    recall  f1-score   support

           0     0.7647    0.6263    0.6886      1437
           1     0.9594    0.9786    0.9689     12960

    accuracy                         0.9435     14397
   macro avg     0.8620    0.8025    0.8288     14397
weighted avg     0.9399    0.9435    0.9409     14397



## Local relational factual features

V1 used whole-document presence checks, which were weak for relational errors. V2 computes number, polarity, temporal, and high-IDF consistency **inside the local span the model itself considers strongest**.

In [14]:
def best_span_features(df, best_indices):
    rows = []
    for pos, r in enumerate(tqdm(df.itertuples(), total=len(df), desc="Best-span relational features")):
        k = int(best_indices[pos])
        k = min(max(k, 0), N_LOCAL_VIEWS - 1)
        span = getattr(r, f"local_{k}")
        tt = tokens(r.title_model); tset = set(tt); sset = set(tokens(span))
        idf_denom = sum(tok_idf(t) for t in tset) + 1e-9
        idf_cov = sum(tok_idf(t) for t in tset if t in sset) / idf_denom
        high = [t for t in tset if tok_idf(t) >= HIGH_IDF_THRESHOLD and not t.isdigit()]
        high_missing = sum(t not in sset for t in high) / max(1, len(high))

        tnums = {canonical_number(x) for x in raw_numbers(r.title_model)}
        snums = {canonical_number(x) for x in raw_numbers(span)}
        num_missing = sum(n not in snums for n in tnums) / max(1, len(tnums))

        polarity = [t for t in tt if t in POLARITY_WORDS]
        polarity_missing = sum(t not in sset for t in polarity) / max(1, len(polarity))
        temporal = [t for t in tt if t in TEMPORAL_WORDS]
        temporal_missing = sum(t not in sset for t in temporal) / max(1, len(temporal))

        title_entities = [normalize_key(e) for e in ENTITY_RE.findall(r.title_model) if e not in ENTITY_STOP]
        span_key = normalize_key(span)
        entity_missing = sum(e not in span_key for e in title_entities) / max(1, len(title_entities))
        rows.append({
            "best_span_idf_cov": idf_cov,
            "best_span_high_idf_missing": high_missing,
            "best_span_num_missing": num_missing,
            "best_span_polarity_missing": polarity_missing,
            "best_span_temporal_missing": temporal_missing,
            "best_span_entity_missing": entity_missing,
        })
    return pd.DataFrame(rows, index=df.index)

train_rel = best_span_features(train, oof_model.best_local_idx.astype(int).to_numpy())

# For test, compute relation features per fold using that fold's chosen local span, then average.
test_rel_folds = []
for tf in test_fold_features:
    rel = best_span_features(test, tf.loc[test.index, "best_local_idx"].astype(int).to_numpy())
    test_rel_folds.append(rel)
test_rel = sum(test_rel_folds) / len(test_rel_folds)

Best-span relational features:   0%|          | 0/14397 [00:00<?, ?it/s]

Best-span relational features:   0%|          | 0/3603 [00:00<?, ?it/s]

Best-span relational features:   0%|          | 0/3603 [00:00<?, ?it/s]

Best-span relational features:   0%|          | 0/3603 [00:00<?, ?it/s]

## Compact cross-fitted meta model

The meta layer intentionally stays small. It consumes calibrated semantic support, the complete global/local NLI state, local/global disagreement, and relational checks on the best local span. It does not reintroduce the broad V1 feature set that showed shortcut/collinearity signs.

In [15]:
META_FEATURES = [
    "semantic_cal",
    "global_e", "global_n", "global_c",
    "local_pool_e", "local_max_e", "local_top2_e", "local_max_n", "local_max_c",
    "best_local_e", "best_local_n", "best_local_c", "global_local_gap",
    "best_span_idf_cov", "best_span_high_idf_missing", "best_span_num_missing",
    "best_span_polarity_missing", "best_span_temporal_missing", "best_span_entity_missing",
    "retrieval_gain", "log_content_words", "content_quality_short",
]

X_meta = pd.concat([
    oof_model.drop(columns=["best_local_idx"]), train_rel, train_static
], axis=1)[META_FEATURES]
X_test_meta = pd.concat([
    test_model, test_rel, test_static
], axis=1)[META_FEATURES]
assert np.isfinite(X_meta.to_numpy(dtype=float)).all() and np.isfinite(X_test_meta.to_numpy(dtype=float)).all()


def make_meta_model():
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(C=1.0, max_iter=1200, solver="liblinear", random_state=SEED)
    )

oof_meta = np.full(len(train), np.nan, dtype=float)
for fold, (tr_idx, va_idx) in enumerate(splits):
    m = make_meta_model()
    m.fit(X_meta.iloc[tr_idx], train.label.values[tr_idx])
    oof_meta[va_idx] = m.predict_proba(X_meta.iloc[va_idx])[:, 1]
assert np.isfinite(oof_meta).all()

meta_diag = probability_diagnostics(train.label.values, oof_meta, prefix="V2 cross-fitted meta OOF")
meta_t = meta_diag["threshold"]
print(classification_report(train.label.values, (oof_meta >= meta_t).astype(int), digits=4))

meta = make_meta_model(); meta.fit(X_meta, train.label.values)
test_meta = meta.predict_proba(X_test_meta)[:, 1]

V2 cross-fitted meta OOF macro_f1=0.836466 t=0.596983 F1_0=0.7047 F1_1=0.9682
V2 cross-fitted meta OOF p[min,p01,p10,p50,p90,p99,max]=0.02447, 0.06144, 0.65547, 0.97891, 0.99130, 0.99459, 0.99896
V2 cross-fitted meta OOF confusion_matrix=[[987, 450], [377, 12583]]
              precision    recall  f1-score   support

           0     0.7236    0.6868    0.7047      1437
           1     0.9655    0.9709    0.9682     12960

    accuracy                         0.9426     14397
   macro avg     0.8445    0.8289    0.8365     14397
weighted avg     0.9413    0.9426    0.9419     14397



## Error-slice validation

These are the V1 bottlenecks that V2 is explicitly designed to improve. Slice scores use the single strict OOF meta threshold, not per-slice threshold tuning.

In [16]:
def fixed_threshold_metrics(name, mask, y, p, threshold):
    mask = np.asarray(mask, dtype=bool)
    if mask.sum() == 0:
        return None
    yt = np.asarray(y)[mask]; pt = np.asarray(p)[mask]
    pred = (pt >= threshold).astype(int)
    f1s = f1_score(yt, pred, labels=[0,1], average=None, zero_division=0)
    return {
        "slice": name, "rows": int(mask.sum()), "negatives": int((yt==0).sum()),
        "macro_f1": float(f1_score(yt, pred, average="macro", zero_division=0)),
        "f1_0": float(f1s[0]), "f1_1": float(f1s[1]),
    }

y = train.label.values
slice_specs = {
    "overall": np.ones(len(train), dtype=bool),
    "content_le_512": train.content_words.values <= 512,
    "content_gt_512": train.content_words.values > 512,
    "content_gt_1000": train.content_words.values > 1000,
    "retrieval_gain_gt_0.4": train.retrieval_gain.values > 0.4,
    "title_has_negation": train_static.title_has_negation.values.astype(bool),
    "high_lexical_overlap": train_static.full_idf_cov.values >= 0.85,
}
slice_rows = [fixed_threshold_metrics(name, mask, y, oof_meta, meta_t) for name, mask in slice_specs.items()]
slice_metrics = pd.DataFrame([r for r in slice_rows if r is not None])
display(slice_metrics)

,slice,rows,negatives,macro_f1,f1_0,f1_1
0,overall,14397,1437,0.836466,0.704748,0.968184
1,content_le_512,13345,1284,0.840692,0.711324,0.970060
2,content_gt_512,1052,153,0.795795,0.648464,0.943125
3,content_gt_1000,101,12,0.751232,0.571429,0.931034
4,retrieval_gain_gt_0.4,126,19,0.586735,0.357143,0.816327
5,title_has_negation,1648,140,0.769723,0.578571,0.960875
6,high_lexical_overlap,4494,246,0.826357,0.670913,0.981801


## Confidence-gated V2 graph layer

Final decision order:

1. **Exact known pair** → deterministic known label.
2. **Unique canonical positive title for this content** + different query title + non-malformed content → deterministic `0`.
3. **Same title known positive with another content** → force `0` only when OOF meta confidence falls below a learned high-precision gate.
4. Otherwise use calibrated V2 meta prediction.

The title gate is selected cross-fitted for the graph-like train diagnostic and on all OOF title-candidates for final test inference.

In [17]:
def build_graph_reference(ref_df):
    pair_to_label = {}
    for key, g in ref_df.groupby("pair_key"):
        labs = set(map(int, g.label))
        if len(labs) == 1:
            pair_to_label[key] = next(iter(labs))
    content_to_pos_titles = defaultdict(set)
    title_to_pos_contents = defaultdict(set)
    for r in ref_df.loc[ref_df.label == 1].itertuples():
        content_to_pos_titles[r.content_key].add(r.title_key)
        title_to_pos_contents[r.title_key].add(r.content_key)
    return pair_to_label, content_to_pos_titles, title_to_pos_contents


def select_title_gate(y, probs, base_threshold, min_precision=TITLE_GATE_MIN_PRECISION):
    y = np.asarray(y, dtype=int); p = np.asarray(probs, dtype=float)
    if len(y) < 5 or len(np.unique(y)) < 2:
        return float(base_threshold), {"macro_f1": np.nan, "forced_precision": np.nan, "forced_rows": 0}
    candidates = np.unique(np.r_[base_threshold, p])
    best = (float(base_threshold), -1.0, 0.0, 0)
    for g in candidates:
        forced = p < g
        if forced.sum() == 0:
            continue
        precision0 = float((y[forced] == 0).mean())
        if precision0 < min_precision:
            continue
        pred = (p >= base_threshold).astype(int)
        pred[forced] = 0
        score = f1_score(y, pred, average="macro", zero_division=0)
        # prefer score, then higher forced precision, then smaller coverage if tied
        key = (score, precision0, -int(forced.sum()))
        if key > (best[1], best[2], -best[3]):
            best = (float(g), float(score), precision0, int(forced.sum()))
    return best[0], {"macro_f1": best[1], "forced_precision": best[2], "forced_rows": best[3]}

# Candidate rows after exact/unique-content LOO logic.
title_candidates = graph_audit[graph_audit.rule == "title_candidate"].index.to_numpy(dtype=int)
full_title_gate, full_title_gate_info = select_title_gate(
    train.loc[title_candidates, "label"].values,
    oof_meta[title_candidates], meta_t
)
print("final title gate:", full_title_gate, full_title_gate_info)

# Cross-fitted gates for a fair graph-like LOO diagnostic.
cf_title_gate_by_fold = {}
for fold, (_, va_idx) in enumerate(splits):
    train_candidates = np.intersect1d(title_candidates, np.setdiff1d(np.arange(len(train)), va_idx))
    gate, info = select_title_gate(train.loc[train_candidates, "label"].values, oof_meta[train_candidates], meta_t)
    cf_title_gate_by_fold[fold] = gate
    print(f"fold={fold} title_gate={gate:.6f} info={info}")

# Build graph-like LOO predictions without contaminating strict semantic CV.
graph_like_pred = (oof_meta >= meta_t).astype(np.int8)
graph_like_rule = np.array(["semantic"] * len(train), dtype=object)
for idx, r in graph_audit.iterrows():
    idx = int(idx)
    if r.rule == "exact_pair":
        graph_like_pred[idx] = int(r.pred); graph_like_rule[idx] = "exact_pair"
    elif r.rule == "unique_content_different_title" and UNIQUE_CONTENT_RULE_ENABLED:
        graph_like_pred[idx] = 0; graph_like_rule[idx] = "unique_content_different_title"

for fold, (_, va_idx) in enumerate(splits):
    gate = cf_title_gate_by_fold[fold]
    va_candidates = np.intersect1d(title_candidates, va_idx)
    force = va_candidates[oof_meta[va_candidates] < gate]
    graph_like_pred[force] = 0
    graph_like_rule[force] = "gated_title_different_content"

graph_like_f1 = f1_score(train.label.values, graph_like_pred, average="macro", zero_division=0)
graph_like_f1s = f1_score(train.label.values, graph_like_pred, labels=[0,1], average=None, zero_division=0)
print(f"V2 graph-like LOO macro_f1={graph_like_f1:.6f} F1_0={graph_like_f1s[0]:.4f} F1_1={graph_like_f1s[1]:.4f}")
display(pd.Series(graph_like_rule).value_counts().to_frame("rows"))

final title gate: 0.9839511586057019 {'macro_f1': 1.0, 'forced_precision': 1.0, 'forced_rows': 32}
fold=0 title_gate=0.985844 info={'macro_f1': 1.0, 'forced_precision': 1.0, 'forced_rows': 17}
fold=1 title_gate=0.983951 info={'macro_f1': 1.0, 'forced_precision': 1.0, 'forced_rows': 23}
fold=2 title_gate=0.983951 info={'macro_f1': 1.0, 'forced_precision': 1.0, 'forced_rows': 24}
V2 graph-like LOO macro_f1=0.924880 F1_0=0.8654 F1_1=0.9843


,rows
semantic,9834
exact_pair,3620
unique_content_different_title,909
gated_title_different_content,34


In [18]:
def apply_graph_v2(query_df, probs, ref_df, title_gate):
    probs = np.asarray(probs, dtype=float).copy()
    pair_to_label, content_to_pos_titles, title_to_pos_contents = build_graph_reference(ref_df)
    rules = np.array(["semantic"] * len(query_df), dtype=object)
    forced = np.full(len(query_df), -1, dtype=np.int8)

    for pos, r in enumerate(query_df.itertuples()):
        # 1) exact pair
        if r.pair_key in pair_to_label:
            lab = int(pair_to_label[r.pair_key])
            forced[pos] = lab; probs[pos] = float(lab); rules[pos] = "exact_pair"
            continue

        malformed = int(r.content_words) < MIN_GRAPH_CONTENT_WORDS
        pos_titles = content_to_pos_titles.get(r.content_key, set())
        # 2) only a genuinely unique canonical positive title is safe to hard-force.
        if (
            UNIQUE_CONTENT_RULE_ENABLED and not malformed
            and len(pos_titles) == 1 and r.title_key not in pos_titles
        ):
            forced[pos] = 0; probs[pos] = 0.0; rules[pos] = "unique_content_different_title"
            continue

        # 3) same-title mismatch is soft/gated and abstains for malformed content.
        pos_contents = title_to_pos_contents.get(r.title_key, set())
        if not malformed and pos_contents and r.content_key not in pos_contents and probs[pos] < title_gate:
            forced[pos] = 0; probs[pos] = 0.0; rules[pos] = "gated_title_different_content"

    labels = (probs >= meta_t).astype(np.int8)
    m = forced >= 0
    labels[m] = forced[m]
    return probs, labels, rules, forced

final_test_prob, final_test_label, test_rule, test_forced = apply_graph_v2(test, test_meta, train, full_title_gate)
print("Test graph coverage:")
display(pd.Series(test_rule).value_counts().to_frame("rows"))
print("Forced labels:", int((test_forced >= 0).sum()))
print("Prediction distribution:", pd.Series(final_test_label).value_counts().sort_index().to_dict())

Test graph coverage:


,rows
semantic,2447
exact_pair,927
unique_content_different_title,212
gated_title_different_content,17


Forced labels: 1156
Prediction distribution: {0: 369, 1: 3234}


## Save V2 artifacts

`submission_v2.csv` is the Kaggle submission file. The other artifacts are intentionally verbose enough to diagnose whether V2 improved the exact V1 failure slices before deciding on any later ensemble or higher-fold run.

In [19]:
submission = sample[["id"]].copy()
assert submission.id.tolist() == test.id.tolist(), "sample_submission order does not match test.csv"
submission["label"] = final_test_label.astype(int)

submission_path = OUTPUT_DIR / "submission_v2.csv"
oof_path = OUTPUT_DIR / "oof_v2.csv"
test_scores_path = OUTPUT_DIR / "test_scores_v2.csv"
history_path = OUTPUT_DIR / "training_history_v2.csv"
slice_path = OUTPUT_DIR / "slice_metrics_v2.csv"
graph_path = OUTPUT_DIR / "graph_diagnostics_v2.csv"
config_path = OUTPUT_DIR / "config_v2.json"

submission.to_csv(submission_path, index=False)

oof_export = pd.DataFrame({
    "id": train.id,
    "label": train.label.astype(int),
    "semantic_raw": oof_model.semantic_raw.values,
    "semantic_cal": oof_model.semantic_cal.values,
    "meta_prob": oof_meta,
    "meta_label": (oof_meta >= meta_t).astype(int),
    "graph_like_label": graph_like_pred,
    "graph_like_rule": graph_like_rule,
})
for col in ["global_e","global_n","global_c","local_pool_e","local_max_e","local_max_c","global_local_gap"]:
    oof_export[col] = oof_model[col].values
oof_export.to_csv(oof_path, index=False)

test_export = pd.DataFrame({
    "id": test.id,
    "semantic_raw": test_model.semantic_raw.values,
    "semantic_cal": test_model.semantic_cal.values,
    "meta_prob": test_meta,
    "final_prob": final_test_prob,
    "graph_rule": test_rule,
    "forced_label": test_forced,
    "label": final_test_label.astype(int),
})
for col in ["global_e","global_n","global_c","local_pool_e","local_max_e","local_max_c","global_local_gap"]:
    test_export[col] = test_model[col].values
test_export.to_csv(test_scores_path, index=False)

pd.DataFrame(train_history).to_csv(history_path, index=False)
slice_metrics.to_csv(slice_path, index=False)

graph_diag = pd.DataFrame({
    "metric": [
        "unique_content_rule_enabled", "unique_content_loo_rows", "unique_content_loo_accuracy",
        "title_candidate_rows", "final_title_gate", "graph_like_macro_f1",
        "strict_semantic_macro_f1", "strict_meta_macro_f1",
    ],
    "value": [
        int(UNIQUE_CONTENT_RULE_ENABLED), len(unique_rows),
        float((unique_rows.pred == unique_rows.y).mean()) if len(unique_rows) else np.nan,
        len(title_candidates), full_title_gate, graph_like_f1,
        semantic_diag["macro_f1"], meta_diag["macro_f1"],
    ]
})
graph_diag.to_csv(graph_path, index=False)

with open(config_path, "w") as f:
    json.dump({
        "version": "v2",
        "model": MODEL_NAME,
        "n_folds": N_FOLDS,
        "epochs": EPOCHS,
        "global_max_length": GLOBAL_MAX_LENGTH,
        "local_max_length": LOCAL_MAX_LENGTH,
        "n_local_views": N_LOCAL_VIEWS,
        "batch_size": BATCH_SIZE,
        "eval_batch_size": EVAL_BATCH_SIZE,
        "grad_accum": GRAD_ACCUM,
        "lr": LR,
        "neg_class_weight": NEG_CLASS_WEIGHT,
        "pairwise_lambda": PAIRWISE_LAMBDA,
        "synthetic_counterfactuals": USE_SYNTHETIC_COUNTERFACTUALS,
        "synthetic_rate": SYNTHETIC_RATE,
        "synthetic_lambda": SYNTHETIC_LAMBDA,
        "unique_content_rule_min_precision": UNIQUE_CONTENT_RULE_MIN_PRECISION,
        "title_gate_min_precision": TITLE_GATE_MIN_PRECISION,
        "meta_threshold": meta_t,
        "title_gate": full_title_gate,
        "seed": SEED,
    }, f, indent=2)

print("Saved:")
for p in [submission_path, oof_path, test_scores_path, history_path, slice_path, graph_path, config_path]:
    print(" -", p)

Saved:
 - /kaggle/working/submission_v2.csv
 - /kaggle/working/oof_v2.csv
 - /kaggle/working/test_scores_v2.csv
 - /kaggle/working/training_history_v2.csv
 - /kaggle/working/slice_metrics_v2.csv
 - /kaggle/working/graph_diagnostics_v2.csv
 - /kaggle/working/config_v2.json


## Reading the V2 result

Submit **`submission_v2.csv`** first.

Before interpreting leaderboard movement, compare V2 against the V1 diagnostics that motivated this design:

- strict connected-component OOF Macro F1 and class-0 F1,
- `content_gt_512`, `content_gt_1000`, and `retrieval_gain_gt_0.4`,
- `title_has_negation` and `high_lexical_overlap`,
- graph-like LOO diagnostic and rule coverage,
- test prediction distribution and graph-rule distribution.

The first V2 run should remain **3 folds × 2 epochs**. A backbone swap, XLM-R ensemble, multi-seed ensemble, or 5-fold run is intentionally **not** included here because those were later experiments, not part of this targeted V2 implementation.